In [1]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ===========================
# 1. 数据处理 (ETL Pipeline) - 这是面试官最看重的部分
# ===========================
print("正在读取 PDF 文件...")

# --- 修改这里：换成你的 PDF 文件路径 ---
pdf_path = "./data/stat_book.pdf"  # 假设你把文件放在 data 文件夹下
# -------------------------------------

if not os.path.exists(pdf_path):
    print(f"报错：找不到文件 {pdf_path}，请检查路径！")
    exit()

# A. 加载 (Load)
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"成功加载 PDF，共 {len(docs)} 页。")

# B. 切分 (Split) - 统计学专业的你主要研究这里
# 为什么是 500？为什么重叠 50？这都是可以优化的参数
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # 每一块的大小（字符数）
    chunk_overlap=50,     # 前后重叠 50 个字，防止句子被切断
    separators=["\n\n", "\n", "。", "！", "？", "，"] # 优先按段落切，其次按句子切
)

all_splits = text_splitter.split_documents(docs)
print(f"切分完成：将文档切成了 {len(all_splits)} 个片段 (Chunks)。")

# ===========================
# 2. 向量化与存储
# ===========================
print("正在构建向量数据库 (这可能需要一点时间)...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")
vector_store = FAISS.from_documents(all_splits, embeddings)
print(">>> 知识库构建完成！")

# ===========================
# 3. 检索与问答 (和之前一样)
# ===========================
API_KEY = "sk-43401194122e4d4f8645889b070af0df" # 记得填你的 Key
BASE_URL = "https://api.deepseek.com"

llm = ChatOpenAI(
    model="deepseek-chat", 
    api_key=API_KEY,
    base_url=BASE_URL, 
    temperature=0.1
)

prompt = ChatPromptTemplate.from_template("""
你是一个做事很认真的助教。基于【已知信息】回答【问题】。
如果无法从已知信息中得到答案，请说“根据提供的文档无法回答此问题”。

【已知信息】:
{context}

【问题】:
{question}
""")

chain = prompt | llm | StrOutputParser()

# --- 测试 ---
# 问一个你那个 PDF 里才有的问题
question = "请总结一下这份文档主要讲了什么？" 

print(f"\n正在根据文档回答: {question} ...")
related_docs = vector_store.similarity_search(question, k=3) # 找最相关的3段
context = "\n\n".join([d.page_content for d in related_docs])
answer = chain.invoke({"context": context, "question": question})

print("\n=== AI 回答 ===")
print(answer)

正在读取 PDF 文件...
成功加载 PDF，共 25 页。
切分完成：将文档切成了 229 个片段 (Chunks)。
正在构建向量数据库 (这可能需要一点时间)...


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>>> 知识库构建完成！

正在根据文档回答: 请总结一下这份文档主要讲了什么？ ...

=== AI 回答 ===
根据提供的文档，这份文档主要讨论了神经ODE（常微分方程）的表达能力，并提出了一个新的ODE模型以提升其表达能力和性能。此外，文档还涉及了衡量表达能力的方法，并提到了两个具体的数据集任务：一个是预测家庭每小时的有功功率消耗，另一个是预测每小时交通流量。文档中简要描述了这些任务的数据预处理步骤和特征选择。
